In [ ]:
from openai import OpenAI
from opensearchpy import OpenSearch
import torch
from sentence_transformers import SentenceTransformer 
import json
import time
from tqdm import tqdm
import pandas as pd
import os
import random
from collections import Counter

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
model = SentenceTransformer("intfloat/multilingual-e5-base", device=device)

In [ ]:
client = OpenSearch(
    hosts=[{'host': 'seu-host.com.br', 'port': 9200}],
    http_auth=('xxxxx', 'xxxxx'),
    use_ssl=True,
    verify_certs=False, 
    timeout=30)

In [ ]:
def run_query(query):
    query = 'query: ' + query

    emb_qry = model.encode(query, show_progress_bar=False)
    emb_qry = emb_qry.tolist()

    query_body = {"size": 20,
        "query": {"knn": {"embedding": {"vector": emb_qry, "k": 10}}},
        "_source": False,
            "fields": ["docid", "json_name", "text"],
    }
    
    response = client.search(
        body = query_body,
        index = 'regis1'
    )
    chunks = []
    for j, hit in enumerate(response["hits"]["hits"]):
        chunks.append({
            'index': hit['_index'],
            'order': j,
            'id': hit['_id'],
            'score': hit['_score'],
            'chunk': hit['fields']['text'][0]
        })
    return chunks

In [ ]:
tst = run_query("Onde fica a bacia de pelotas?")

In [ ]:
def generate_llm_response(prompt, model):

    client = openai.OpenAI(
        api_key= "sk-XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
    )
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role":"user","content": prompt},
        ]
    )
    return completion.choices[0].message.content

def make_prompt(prompt_file, completion, model='gpt-4o-mini'):
    with open(prompt_file) as pf:
        pre_prompt = pf.read()
        return generate_llm_response(pre_prompt+completion,model)

In [ ]:
def try_to_decode_json(str):
    try:
        ans = json.loads(str)
    except:
        ans = json.loads(str[8:-4])
    return ans

In [ ]:
import math

def is_random(s: str) -> bool:
    """
    Classifica uma string com base em algumas heurísticas:
      - Proporção de caracteres alfabéticos.
      - Proporção de espaços (para indicar a presença de palavras).
      - Validação das palavras encontradas (se possuem pelo menos uma vogal).
      - Entropia da string (para medir o grau de aleatoriedade).
    
    Retorna um dicionário com os valores calculados e a decisão final.
    A decisão final pode ser "texto" ou "aleatório" baseada em thresholds definidos.
    """
    resultado = {}
    total = len(s)
    
    if total == 0:
        resultado['final'] = "aleatório"  # ou pode tratar caso especial de string vazia
        return resultado

    # Heurística 1: Proporção de caracteres alfabéticos
    count_alpha = sum(1 for c in s if c.isalpha())
    ratio_alpha = count_alpha / total
    resultado['ratio_alpha'] = ratio_alpha

    # Heurística 2: Proporção de espaços (indicativo de separação de palavras)
    count_spaces = s.count(" ")
    ratio_spaces = count_spaces / total
    resultado['ratio_spaces'] = ratio_spaces

    # Heurística 3: Verificação das palavras válidas (contendo ao menos uma vogal)
    palavras = s.split()
    validas = 0
    vogais = "aeiouAEIOUáéíóúÁÉÍÓÚâêîôûÂÊÎÔÛãõÃÕ"
    for palavra in palavras:
        if any(letra in vogais for letra in palavra):
            validas += 1
    ratio_validas = validas / len(palavras) if palavras else 0
    resultado['ratio_validas'] = ratio_validas

    # Heurística 4: Cálculo da entropia da string (medida de aleatoriedade)
    freq = {}
    for char in s:
        freq[char] = freq.get(char, 0) + 1
    entropia = -sum((count / total) * math.log2(count / total) for count in freq.values())
    resultado['entropia'] = entropia

    # Decisão final: definir thresholds para as heurísticas.
    # Estes thresholds podem ser ajustados conforme o domínio de aplicação.
    # Exemplo:
    # - Se a proporção de letras for alta (ex: > 0.6),
    # - Se a maioria das palavras tiver pelo menos uma vogal (ex: > 0.5),
    # - Se a entropia estiver dentro de um intervalo "típico" para texto (ex: entre 3.0 e 5.5)
    if ratio_alpha > 0.6 and ratio_validas > 0.5 and (3.0 <= entropia <= 5.5):
        resultado['final'] = "texto"
    else:
        resultado['final'] = "aleatório"
    
    return resultado['final'] == "aleatório"



In [ ]:
def select_chunks(chunks, strategy="top2"):

    if strategy == "top2":
        return chunks[:2]
    else:
        return random.sample(chunks,2)

def make_composite_queries_from_seed2(seed_query, selection):

    chunks = run_query(seed_query)
    chunk1, chunk2 = select_chunks(chunks,selection)

    if is_random(chunk1['chunk']) or is_random(chunk2['chunk']):
        return {
            'question': "INVALIDA",
            'answer': "INVALIDA",
            'chunk1_id': chunk1['id'],
            'chunk2_id': chunk2['id'],
            'chunk1': chunk1['chunk'],
            'chunk2': chunk2['chunk'],
            "justificativa": "INVALIDA",
            'assunto': seed_query,
            "selection":selection
    }  

    time.sleep(3)
    response = make_prompt("prompts/composeN.txt",
                           f"Documento 1: {{{chunk1['chunk']}}}\n\nDocumento 2: {{{chunk2['chunk']}}}",
                           model='gpt-4o')
    
    question = try_to_decode_json(response)
            
    return {
        'question': question['pergunta'],
        'answer': question['resposta'],
        'chunk1_id': chunk1['id'],
        'chunk2_id': chunk2['id'],
        'chunk1': chunk1['chunk'],
        'chunk2': chunk2['chunk'],
        "justificativa": question['justificativa'],          
        'assunto': seed_query,
        "selection":selection
    }    

In [ ]:
seeds = [
    "Quais são os principais tipos de trapas estruturais?",
    "Quais indicadores geoquímicos são utilizados para avaliar a maturação térmica em rochas geradoras?",
    "Como a diagênese impacta a distribuição de fluidos em reservatórios?",
    "Como a deposição dos prismas clásticos influencia a formação de potenciais reservatórios?",
    "Como a heterogeneidade interna dos depósitos turbidíticos impacta a conectividade dos reservatórios?"
]

In [ ]:
tst = make_composite_queries_from_seed2(seeds[0], 'top2')

In [ ]:
ans = []
for i, seed in tqdm(enumerate(seeds)):
    ans.append(make_composite_queries_from_seed2(seed, "top2"))
    for j in tqdm(range(4)):
        ans.append(make_composite_queries_from_seed2(seed, "random"))

In [ ]:
pd.DataFrame.from_records(ans).to_csv("25_compostas_novo.csv", index=False)